# Build ArcGIS Pro deliverable bundle

Produces a self-contained folder `ArcGIS_deliverable/` at the repo root, ready to share
with ArcGIS Pro users.

For each output layer the notebook:
- Reads the renamed source `.gpkg` from `intermediate_results/geopackages/` or `results/`.
- Computes a categorical class column from the chosen attribute (quintile, log-quintile,
  fixed bins, binary, or pre-existing class).
- Writes a paired `.gpkg` + `.lyrx` into `ArcGIS_deliverable/<sub-folder>/`. The `.lyrx`
  references the `.gpkg` by **relative path** so the whole folder is portable.
- Also writes a master `serbia_criticality_atlas.lyrx` at the bundle root that wraps
  every layer in nested `CIMGroupLayer`s mirroring `layer_inventory.md`.

Special handling for `network_hazard_exposure`: a single `.gpkg` with 4 class columns
(flood depth / snow drift / pavement temperature / wildfire susceptibility) and a single
`.lyrx` that exposes those four attributes as 4 sub-layers under a group.

> Runs end-to-end on any OS with `geopandas` — does NOT require `arcpy`.


In [ ]:
import json
import shutil
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

def _find_repo() -> Path:
    """Walk upward from cwd to find the repo root (contains intermediate_results/)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "intermediate_results").is_dir() and (p / "notebooks").is_dir():
            return p
    raise FileNotFoundError(
        "Could not find repo root from "
        + str(Path.cwd())
        + " (looking for a parent containing both intermediate_results/ and notebooks/)"
    )

REPO = _find_repo()
SRC_INTERMEDIATE = REPO / "intermediate_results" / "geopackages"
SRC_RESULTS = REPO / "results"
BUNDLE = REPO / "ArcGIS_deliverable"

SUB_FOLDERS = ("input", "hazard_exposure", "travel_disruption", "local_accessibility", "final_results")

BUNDLE.mkdir(parents=True, exist_ok=True)
for sub in SUB_FOLDERS:
    (BUNDLE / sub).mkdir(exist_ok=True)

# Tidy: remove a stale `results/` sub-folder from a previous build, if present.
stale = BUNDLE / "results"
if stale.is_dir():
    shutil.rmtree(stale)

print(f"Repo:   {REPO}")
print(f"Bundle: {BUNDLE}")


## Preprocess raw inputs (OSM + PERS)

The Input group of the atlas (OSM and PERS road networks) is generated here
directly from the raw files in `input_files/`. The resulting `.gpkg`s live in
`intermediate_results/geopackages/input/` and are then picked up by the regular
build loop just like any other layer.

This step is idempotent — if the two input `.gpkg`s already exist, it is
skipped (parsing the OSM `.pbf` is slow).


In [ ]:
import re

INPUT_DATA = REPO / "input_files"
INPUT_OUT  = REPO / "intermediate_results" / "geopackages" / "input"
INPUT_OUT.mkdir(parents=True, exist_ok=True)

OSM_OUT  = INPUT_OUT / "osm_road_network.gpkg"
PERS_OUT = INPUT_OUT / "pers_road_network.gpkg"


def _map_osm_category(highway: str) -> str:
    if highway in ("motorway", "motorway_link"):  return "motorway"
    if highway in ("trunk", "trunk_link"):        return "trunk"
    if highway in ("primary", "primary_link"):    return "primary"
    if highway in ("secondary", "secondary_link"):return "secondary"
    if highway in ("tertiary", "tertiary_link"):  return "tertiary"
    if highway in ("residential", "road", "unclassified", "track"): return "other"
    return highway


def _extract_tag(other_tags: str, key: str) -> str | None:
    if other_tags is None:
        return None
    m = re.search(rf'"{key}"=>"([^"]+)"', other_tags)
    return m.group(1) if m else None


def build_osm_road_network(pbf_path: Path, out_path: Path) -> None:
    """Filter the OSM lines layer to driveable highways and assign a 6-class category."""
    features = gpd.read_file(pbf_path, layer="lines")
    features = features[features["highway"].notna()]
    for key in ("highway", "name", "maxspeed", "oneway", "lanes", "surface"):
        if key not in features.columns:
            features[key] = features["other_tags"].apply(lambda x: _extract_tag(x, key))

    road_types = [
        "primary", "trunk", "motorway", "motorway_link", "trunk_link", "primary_link",
        "secondary", "secondary_link", "tertiary", "tertiary_link",
        "residential", "road", "unclassified", "track",
    ]
    osm = features[features["highway"].isin(road_types)].reset_index(drop=True)
    osm = osm[["osm_id", "highway", "name", "maxspeed", "oneway", "lanes", "surface", "geometry"]].copy()
    osm["road_category"] = osm["highway"].apply(_map_osm_category)

    if out_path.exists():
        out_path.unlink()
    osm.to_file(str(out_path), driver="GPKG", layer="osm_road_network")
    print(f"  Wrote {out_path}  ({len(osm):,} features)")


def build_pers_road_network(shp_path: Path, out_path: Path) -> None:
    """Filter the PERS shapefile to the 5 official road categories."""
    gdf = gpd.read_file(shp_path)
    # Source has duplicated column names — keep first occurrence.
    gdf = gdf.loc[:, ~gdf.columns.duplicated()].copy()
    categories = ["IA", "IM", "IB", "IIA", "IIB"]
    pers = gdf[gdf["kategorija"].isin(categories)].copy()
    if out_path.exists():
        out_path.unlink()
    pers.to_file(str(out_path), driver="GPKG", layer="pers_road_network")
    print(f"  Wrote {out_path}  ({len(pers):,} features)")


# Idempotent: skip both if outputs already exist.
if OSM_OUT.exists() and PERS_OUT.exists():
    print(f"Input .gpkgs already present — skipping preprocessing.")
    print(f"  {OSM_OUT}")
    print(f"  {PERS_OUT}")
else:
    osm_pbf = INPUT_DATA / "SRB.osm.pbf"
    pers_shp = INPUT_DATA / "DeoniceRSDP-Jul2025..shp"
    if not osm_pbf.exists():
        raise FileNotFoundError(f"OSM source missing: {osm_pbf}")
    if not pers_shp.exists():
        raise FileNotFoundError(f"PERS source missing: {pers_shp}")
    print(f"Building OSM input from {osm_pbf} (this can take a few minutes)...")
    build_osm_road_network(osm_pbf, OSM_OUT)
    print(f"Building PERS input from {pers_shp}...")
    build_pers_road_network(pers_shp, PERS_OUT)


## Palettes, class labels and widths


In [ ]:
# Six-class composite-criticality scale (matches 5d notebook)
CRIT_LABELS = ["No criticality", "Very Low", "Low", "Medium", "High", "Very High"]
W_CRIT_FULL = [0.2, 0.6, 0.9, 1.3, 2.0, 3.0]
W_CRIT_MEAN = [0.2, 0.6, 1.0, 1.5, 2.2, 3.2]
PAL_PURPLE  = ["#e0e0e0", "#edf8fb", "#b3cde3", "#8c96c6", "#8856a7", "#810f7c"]
PAL_MEAN    = ["#e0e0e0", "#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494"]

# Five-class scales (no zero category)
QUINT_LABELS = ["Very Low", "Low", "Medium", "High", "Very High"]
W_5_LIGHT    = [0.5, 0.9, 1.3, 1.9, 2.6]
W_5_FLAT     = [1.2, 1.2, 1.2, 1.2, 1.2]
PT_SIZES     = [4.0, 5.0, 6.0, 7.0, 8.0]

# Hazard severity / criticality colours
PAL_REDS    = ["#fff5f0", "#fcbba1", "#fc9272", "#fb6a4a", "#cb181d"]
PAL_OR_RD   = ["#fef0d9", "#fdcc8a", "#fc8d59", "#e34a33", "#b30000"]
PAL_VIRIDIS = ["#440154", "#3b528b", "#21908c", "#5dc863", "#fde725"]
PAL_ACC     = ["#1a9641", "#a6d96a", "#ffffbf", "#fdae61", "#d7191c"]
PAL_BLUES   = ["#deebf7", "#9ecae1", "#6baed6", "#3182bd", "#08519c"]
PAL_DKBLUES = ["#deebf7", "#bdd7e7", "#6baed6", "#2171b5", "#08306b"]

# Precipitation — diverging blue→red on % change in max 1-day precip
PRECIP_LABELS = ["< -2 %", "-2 to 0 %", "0 to 5 %", "5 to 10 %", "10 to 20 %", "> 20 %"]
PRECIP_BINS   = [-100, -2, 0, 5, 10, 20, 1e6]
PAL_PRECIP    = ["#2171b5", "#6baed6", "#fee0d2", "#fcae91", "#fb6a4a", "#a50f15"]
W_PRECIP      = [1.2, 1.2, 1.2, 1.2, 1.2, 1.2]

# Future flood RP — diverging red→light blue (red = low RP = high risk)
RP_LABELS = ["10–25 yrs", "25–50 yrs", "50–100 yrs", "100–150 yrs", "> 150 yrs"]
RP_BINS   = [0, 25, 50, 100, 150, 1e6]
PAL_RP    = ["#a50f15", "#de2d26", "#fb6a4a", "#fcae91", "#fee5d9"]

# 6-class labels for zero-aware quintile classification of hazard sub-metrics
HAZ6_LABELS_FLOOD = ["No flooding", "Low", "Medium-Low", "Medium", "Medium-High", "High"]
HAZ6_LABELS_SNOW  = ["No drift",    "Low", "Medium-Low", "Medium", "Medium-High", "High"]
HAZ6_W            = [0.2, 0.6, 0.9, 1.3, 2.0, 3.0]
HAZ6_PAL_FLOOD    = ["#cccccc", "#deebf7", "#9ecae1", "#4292c6", "#2171b5", "#08519c"]
HAZ6_PAL_SNOW     = ["#cccccc", "#deebf7", "#bdd7e7", "#6baed6", "#2171b5", "#08306b"]

# Binary (wildfire susceptibility)
BIN_LABELS_FIRE = ["Not susceptible", "Susceptible"]
BIN_PAL_FIRE    = ["#cccccc", "#cb181d"]
BIN_W_FIRE      = [0.4, 1.8]

# Input layers — OSM road network (matches 1a notebook's matplotlib styling)
OSM_LABELS  = ["motorway", "trunk", "primary", "secondary", "tertiary", "other"]
OSM_PAL     = ["#8B0000",   "#1E90FF", "#A52A2A","#FFA500",   "#228B22",   "#ccc5b9"]
OSM_WIDTHS  = [3.0,         2.5,       2.0,       1.5,         1.0,         0.6]

# Input layers — PERS road network (Serbian official categories IA/IM/IB/IIA/IIB)
PERS_LABELS = ["IA",       "IM",       "IB",       "IIA",      "IIB"]
PERS_PAL    = ["#8B0000",  "#1E90FF",  "#A52A2A",  "#FFA500",  "#228B22"]
PERS_WIDTHS = [3.0,        2.5,        2.0,        1.5,        1.0]


## Manifest — one entry per output layer

`method` controls how the class column is derived:
- `precomputed` — use an existing class field as-is.
- `quintile` — 5-class quantile of the field.
- `log_quintile` — zero-aware 5-class quantile (zeros become the first label).
- `log_quintile_6` — zero-aware quintile with **6** labels (zeros get a dedicated label).
- `bins` — fixed-edge classification using `bins`.
- `binary` — values > 0 → second label, otherwise first label.
- `multi` — one .gpkg + one .lyrx exposing several sub-renderers as a group;
  carries an `attributes` list, one entry per sub-renderer.


In [ ]:
def L(src, sub, name, title, field, geom, method, labels, palette, widths, **kw):
    return dict(src=src, out_dir=sub, name=name, title=title,
                field=field, geom=geom, method=method,
                labels=labels, palette=palette, widths=widths, **kw)

# Sub-renderer config for the multi-attribute network_hazard_exposure layer
NETWORK_HAZARD_ATTRIBUTES = [
    {
        "field": "max_depth", "out_field": "flood_class",
        "title": "Network — flood depth",
        "method": "log_quintile_6",
        "labels": HAZ6_LABELS_FLOOD, "palette": HAZ6_PAL_FLOOD, "widths": HAZ6_W,
    },
    {
        "field": "snow_drift", "out_field": "snow_class",
        "title": "Network — snow drift length",
        "method": "log_quintile_6",
        "labels": HAZ6_LABELS_SNOW, "palette": HAZ6_PAL_SNOW, "widths": HAZ6_W,
    },
    {
        "field": "max_pavement_temp", "out_field": "pavement_class",
        "title": "Network — max pavement temperature",
        "method": "quintile",
        "labels": QUINT_LABELS, "palette": PAL_REDS, "widths": W_5_LIGHT,
    },
    {
        "field": "wildfire_risk", "out_field": "wildfire_class",
        "title": "Network — wildfire susceptibility",
        "method": "binary",
        "labels": BIN_LABELS_FIRE, "palette": BIN_PAL_FIRE, "widths": BIN_W_FIRE,
    },
]

LAYERS = [
    # ---- Input (raw road networks from 1a_Network_Figures) ----
    L(SRC_INTERMEDIATE / "input" / "osm_road_network.gpkg",
      "input", "osm_road_network",
      "OSM Road Network — Serbia",
      "road_category", "line", "precomputed",
      OSM_LABELS, OSM_PAL, OSM_WIDTHS),

    L(SRC_INTERMEDIATE / "input" / "pers_road_network.gpkg",
      "input", "pers_road_network",
      "PERS Road Network — Serbia (Putevi Srbije)",
      "kategorija", "line", "precomputed",
      PERS_LABELS, PERS_PAL, PERS_WIDTHS),

    # ---- Hazard exposure ----
    L(SRC_INTERMEDIATE / "hazard_exposure" / "network_hazard_exposure.gpkg",
      "hazard_exposure", "network_hazard_exposure",
      "Network — combined hazard exposure (4 metrics)",
      None, "line", "multi",
      None, None, None,
      # multi-specific:
      attributes=NETWORK_HAZARD_ATTRIBUTES,
      field_aliases={"snow_drift": "dužina_sn"}),

    L(SRC_INTERMEDIATE / "hazard_exposure" / "future_floods_network.gpkg",
      "hazard_exposure", "future_floods_network",
      "Future flood RP on roads (3 °C warming)",
      "rp30_mean", "line", "bins",
      RP_LABELS, PAL_RP, [1.0, 1.2, 1.5, 1.8, 2.2], bins=RP_BINS),

    L(SRC_INTERMEDIATE / "hazard_exposure" / "future_floods.gpkg",
      "hazard_exposure", "future_floods",
      "Future flood RP — HydroBASINS (3 °C warming)",
      "rp30_mean", "polygon", "bins",
      RP_LABELS, PAL_RP, [0.4]*5, bins=RP_BINS),

    *[L(SRC_INTERMEDIATE / "hazard_exposure" / f"future_precipitation_{scen}.gpkg",
        "hazard_exposure", f"future_precipitation_{scen}",
        f"Future precipitation change — {label}",
        "max_rx1day_pct", "line", "bins",
        PRECIP_LABELS, PAL_PRECIP, W_PRECIP, bins=PRECIP_BINS)
      for scen, label in [
          ("rcp45_near_future", "RCP 4.5, 2031–2060"),
          ("rcp45_far_future",  "RCP 4.5, 2071–2100"),
          ("rcp85_near_future", "RCP 8.5, 2031–2060"),
          ("rcp85_far_future",  "RCP 8.5, 2071–2100"),
      ]],

    # ---- National travel disruption ----
    L(SRC_INTERMEDIATE / "travel_disruption" / "network_criticality.gpkg",
      "travel_disruption", "network_criticality",
      "National travel disruption — passenger hours lost",
      "phl", "line", "log_quintile",
      QUINT_LABELS, PAL_VIRIDIS, W_5_LIGHT),

    # ---- Local accessibility — TT impacts (line edges) ----
    *[L(SRC_INTERMEDIATE / "local_accessibility" / f"{stem}.gpkg",
        "local_accessibility", stem, title,
        "travel_time_impact", "line", "quintile",
        QUINT_LABELS, PAL_OR_RD, W_5_LIGHT)
      for stem, title in [
          ("port_tt_impact",        "Travel-time impact — port access"),
          ("agriculture_tt_impact", "Travel-time impact — agriculture / border"),
          ("police_tt_impact",      "Travel-time impact — police response"),
          ("hospital_tt_impact",    "Travel-time impact — hospital access"),
          ("firefighter_tt_impact", "Travel-time impact — fire-service response"),
          ("factory_tt_impact",     "Travel-time impact — factory access"),
      ]],

    # ---- Local accessibility — baseline (points) ----
    L(SRC_INTERMEDIATE / "local_accessibility" / "police_bl_accessibility.gpkg",
      "local_accessibility", "police_bl_accessibility",
      "Baseline accessibility — nearest police station",
      "travel_time_pol", "point", "quintile",
      QUINT_LABELS, PAL_ACC, PT_SIZES),

    L(SRC_INTERMEDIATE / "local_accessibility" / "hospital_bl_accessibility.gpkg",
      "local_accessibility", "hospital_bl_accessibility",
      "Baseline accessibility — nearest hospital",
      "travel_time_hosp", "point", "quintile",
      QUINT_LABELS, PAL_ACC, PT_SIZES),

    L(SRC_INTERMEDIATE / "local_accessibility" / "firefighter_bl_accessibility.gpkg",
      "local_accessibility", "firefighter_bl_accessibility",
      "Baseline accessibility — nearest fire station",
      "travel_time_ff", "point", "quintile",
      QUINT_LABELS, PAL_ACC, PT_SIZES),

    L(SRC_INTERMEDIATE / "local_accessibility" / "factory_bl_accessibility.gpkg",
      "local_accessibility", "factory_bl_accessibility",
      "Baseline accessibility — factory access",
      "avg_access_time", "point", "quintile",
      QUINT_LABELS, PAL_ACC, PT_SIZES),

    L(SRC_INTERMEDIATE / "local_accessibility" / "agriculture_bl_accessibility.gpkg",
      "local_accessibility", "agriculture_bl_accessibility",
      "Baseline accessibility — farm → ports / borders / rail",
      "avg_access_all", "point", "quintile",
      QUINT_LABELS, PAL_ACC, PT_SIZES),

    # ---- Final composite results ----
    L(SRC_RESULTS / "total_climate_criticality_long.gpkg",
      "final_results", "total_climate_criticality_long",
      "Climate criticality — long composite (CC)",
      "CC_class", "line", "precomputed",
      CRIT_LABELS, PAL_PURPLE, W_CRIT_FULL),

    L(SRC_RESULTS / "total_climate_criticality_short.gpkg",
      "final_results", "total_climate_criticality_short",
      "Climate criticality — short composite (equal-weighted mean of H, T, A)",
      "mean_class", "line", "precomputed",
      CRIT_LABELS, PAL_MEAN, W_CRIT_MEAN),

    # ---- Sub-indices — re-use long composite gpkg (no extra .gpkg copy) ----
    L(SRC_RESULTS / "total_climate_criticality_long.gpkg",
      "final_results", "hazard_exposure_index",
      "Hazard exposure sub-index (H)",
      "H_class", "line", "precomputed",
      CRIT_LABELS, PAL_PURPLE, W_CRIT_FULL,
      reuse_gpkg_name="total_climate_criticality_long",
      atlas_group="sub_indices"),
    L(SRC_RESULTS / "total_climate_criticality_long.gpkg",
      "final_results", "travel_disruption_index",
      "Travel-disruption sub-index (T)",
      "T_class", "line", "precomputed",
      CRIT_LABELS, PAL_PURPLE, W_CRIT_FULL,
      reuse_gpkg_name="total_climate_criticality_long",
      atlas_group="sub_indices"),
    L(SRC_RESULTS / "total_climate_criticality_long.gpkg",
      "final_results", "local_accessibility_index",
      "Local-accessibility sub-index (A)",
      "A_class", "line", "precomputed",
      CRIT_LABELS, PAL_PURPLE, W_CRIT_FULL,
      reuse_gpkg_name="total_climate_criticality_long",
      atlas_group="sub_indices"),
]
print(f"{len(LAYERS)} entries in manifest.")


## Helpers — classification + CIM symbol builders


In [ ]:
def classify_series(series: pd.Series, method: str, labels: list[str],
                    bins: list | None = None) -> pd.Series:
    """Return a Series of string class labels aligned to series.index."""
    s = series

    if method == "precomputed":
        return s.astype(str)

    s = s.astype(float)

    if method == "quintile":
        return pd.qcut(s.rank(method="first"), 5, labels=labels).astype(str)

    if method == "log_quintile":
        out = pd.Series(labels[0], index=s.index, dtype=object)
        nz = s > 0
        if nz.any():
            logged = np.log1p(s[nz])
            out.loc[nz] = pd.qcut(logged.rank(method="first"), 5, labels=labels).astype(str)
        return out

    if method == "log_quintile_6":
        # labels[0] is for zeros; labels[1:] is for the 5 quintile bins.
        if len(labels) != 6:
            raise ValueError("log_quintile_6 needs exactly 6 labels")
        out = pd.Series(labels[0], index=s.index, dtype=object)
        nz = s > 0
        if nz.any():
            logged = np.log1p(s[nz])
            out.loc[nz] = pd.qcut(logged.rank(method="first"), 5,
                                   labels=labels[1:]).astype(str)
        return out

    if method == "bins":
        if bins is None:
            raise ValueError("bins method needs the 'bins' kwarg")
        return pd.cut(s, bins=bins, labels=labels, include_lowest=True).astype(str)

    if method == "binary":
        if len(labels) != 2:
            raise ValueError("binary needs exactly 2 labels")
        out = pd.Series(labels[0], index=s.index, dtype=object)
        out.loc[s.fillna(0) > 0] = labels[1]
        return out

    raise ValueError(f"Unknown classify method: {method}")


def classify_single(gdf: gpd.GeoDataFrame, spec: dict) -> pd.Series:
    """Classify the field named in the single-attribute spec."""
    return classify_series(gdf[spec["field"]], spec["method"],
                           spec["labels"], spec.get("bins"))


In [ ]:
def _hex_to_rgb(h):
    h = h.lstrip("#")
    return [int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16), 100]

def _line_symbol(color, width):
    return {
        "type": "CIMSymbolReference",
        "symbol": {
            "type": "CIMLineSymbol",
            "symbolLayers": [{
                "type": "CIMSolidStroke", "enable": True,
                "capStyle": "Round", "joinStyle": "Round",
                "width": float(width),
                "color": {"type": "CIMRGBColor", "values": _hex_to_rgb(color)},
            }],
        },
    }

def _polygon_symbol(color, _width):
    return {
        "type": "CIMSymbolReference",
        "symbol": {
            "type": "CIMPolygonSymbol",
            "symbolLayers": [
                {"type": "CIMSolidStroke", "enable": True,
                 "capStyle": "Round", "joinStyle": "Round", "width": 0.4,
                 "color": {"type": "CIMRGBColor", "values": [80, 80, 80, 100]}},
                {"type": "CIMSolidFill", "enable": True,
                 "color": {"type": "CIMRGBColor", "values": _hex_to_rgb(color)}},
            ],
        },
    }

def _point_symbol(color, size):
    return {
        "type": "CIMSymbolReference",
        "symbol": {
            "type": "CIMPointSymbol",
            "symbolLayers": [{
                "type": "CIMVectorMarker", "enable": True,
                "anchorPointUnits": "Relative",
                "size": float(size),
                "frame": {"xmin": -2, "ymin": -2, "xmax": 2, "ymax": 2},
                "markerGraphics": [{
                    "type": "CIMMarkerGraphic",
                    "geometry": {"curveRings": [[
                        [1.2246e-16, 2],
                        {"a": [[1.2246e-16, 2], [0, 0], 0, 1]},
                    ]]},
                    "symbol": {
                        "type": "CIMPolygonSymbol",
                        "symbolLayers": [
                            {"type": "CIMSolidStroke", "enable": True,
                             "capStyle": "Round", "joinStyle": "Round", "width": 0.5,
                             "color": {"type": "CIMRGBColor", "values": [50, 50, 50, 100]}},
                            {"type": "CIMSolidFill", "enable": True,
                             "color": {"type": "CIMRGBColor", "values": _hex_to_rgb(color)}},
                        ],
                    },
                }],
                "scaleSymbolsProportionally": True,
                "respectFrame": True,
            }],
            "scaleX": 1, "angleAlignment": "Display",
        },
    }

SYMBOL_BUILDER = {"line": _line_symbol, "polygon": _polygon_symbol, "point": _point_symbol}

def build_renderer(geom, field, labels, palette, widths):
    builder = SYMBOL_BUILDER[geom]
    classes = [{
        "type": "CIMUniqueValueClass",
        "label": str(lbl),
        "patch": "Default",
        "symbol": builder(c, w),
        "values": [{"type": "CIMUniqueValue", "fieldValues": [str(lbl)]}],
        "visible": True,
    } for lbl, c, w in zip(labels, palette, widths)]
    return {
        "type": "CIMUniqueValueRenderer",
        "defaultLabel": "<all other values>",
        "defaultSymbolPatch": "Default",
        "defaultSymbol": builder("#cccccc", 0.5),
        "defaultSymbolVisible": False,
        "fields": [field],
        "groups": [{"type": "CIMUniqueValueGroup", "classes": classes, "heading": field}],
        "useDefaultSymbol": False,
        "polygonSymbolColorTarget": "Fill",
    }


def _feature_table(gpkg_relpath: str, layer_name: str) -> dict:
    return {
        "type": "CIMFeatureTable",
        "editable": True,
        "dataConnection": {
            "type": "CIMStandardDataConnection",
            "workspaceConnectionString": f"AUTHENTICATION_MODE=OSA;DATABASE=.\\{gpkg_relpath}",
            "workspaceFactory": "Sql",
            "dataset": f"main.{layer_name}",
            "datasetType": "esriDTFeatureClass",
        },
        "studyAreaSpatialRel": "esriSpatialRelUndefined",
        "searchOrder": "esriSearchOrderSpatial",
    }


def _feature_layer_def(uri: str, name: str, layer_name: str, gpkg_relpath: str,
                       renderer: dict, *, visible: bool = True) -> dict:
    return {
        "type": "CIMFeatureLayer",
        "name": name,
        "uRI": uri,
        "sourceModifiedTime": {"type": "TimeInstant"},
        "useSourceMetadata": True,
        "description": name,
        "expanded": False,
        "layerType": "Operational",
        "showLegends": True,
        "visibility": visible,
        "displayCacheType": "Permanent",
        "maxDisplayCacheAge": 5,
        "showPopups": True,
        "serviceLayerID": -1,
        "refreshRate": -1,
        "refreshRateUnit": "esriTimeUnitsSeconds",
        "featureTable": _feature_table(gpkg_relpath, layer_name),
        "selectable": True,
        "featureCacheType": "Session",
        "displayFiltersType": "ByScale",
        "renderer": renderer,
    }


def _group_layer_def(uri: str, name: str, child_uris: list[str], *,
                     expanded: bool = False, description: str | None = None) -> dict:
    return {
        "type": "CIMGroupLayer",
        "name": name,
        "uRI": uri,
        "sourceModifiedTime": {"type": "TimeInstant"},
        "useSourceMetadata": True,
        "description": description or name,
        "expanded": expanded,
        "layerType": "Operational",
        "showLegends": True,
        "visibility": True,
        "displayCacheType": "Permanent",
        "maxDisplayCacheAge": 5,
        "showPopups": True,
        "serviceLayerID": -1,
        "refreshRate": -1,
        "refreshRateUnit": "esriTimeUnitsSeconds",
        "layers": list(child_uris),
    }


In [ ]:
def write_single_lyrx(lyrx_path: Path, gpkg_filename: str, layer_name: str,
                      title: str, renderer: dict) -> None:
    """Per-layer .lyrx — one CIMFeatureLayer referencing the gpkg by relative path."""
    doc = {
        "type": "CIMLayerDocument",
        "version": "3.0.0",
        "build": 0,
        "layers": [f"CIMPATH=Internal_Map/{layer_name}.json"],
        "layerDefinitions": [
            _feature_layer_def(
                uri=f"CIMPATH=Internal_Map/{layer_name}.json",
                name=title,
                layer_name=layer_name,
                gpkg_relpath=gpkg_filename,
                renderer=renderer,
            )
        ],
    }
    lyrx_path.write_text(json.dumps(doc, indent=2))


def write_multi_lyrx(lyrx_path: Path, gpkg_filename: str, layer_name: str,
                     title: str, sub_specs: list[dict], geom: str) -> None:
    """Multi-attribute .lyrx — one CIMGroupLayer with N sub feature layers, all
    pointing at the same gpkg but rendering different class fields."""
    root_uri = f"CIMPATH=Internal_Map/{layer_name}_root.json"

    child_layer_defs = []
    child_uris = []
    for sub in sub_specs:
        sub_uri = f"CIMPATH=Internal_Map/{layer_name}__{sub['out_field']}.json"
        renderer = build_renderer(
            geom, sub["out_field"],
            sub["labels"], sub["palette"], sub["widths"],
        )
        child_layer_defs.append(_feature_layer_def(
            uri=sub_uri,
            name=sub["title"],
            layer_name=layer_name,
            gpkg_relpath=gpkg_filename,
            renderer=renderer,
            visible=False,
        ))
        child_uris.append(sub_uri)

    group_def = _group_layer_def(
        uri=root_uri, name=title, child_uris=child_uris, expanded=True
    )

    doc = {
        "type": "CIMLayerDocument",
        "version": "3.0.0",
        "build": 0,
        "layers": [root_uri],
        "layerDefinitions": [group_def, *child_layer_defs],
    }
    lyrx_path.write_text(json.dumps(doc, indent=2))


## Build the bundle


In [ ]:
summary = []

for spec in LAYERS:
    src = Path(spec["src"])
    out_dir = BUNDLE / spec["out_dir"]
    name    = spec["name"]
    title   = spec["title"]
    geom    = spec["geom"]

    if not src.exists():
        print(f"  ✗ MISSING source: {src}")
        summary.append({"layer": name, "status": "missing", "field": "-", "geom": geom,
                        "out_dir": spec["out_dir"]})
        continue

    # ---- Re-use branch: a layer that points at an already-written gpkg ----
    if spec.get("reuse_gpkg_name"):
        existing = spec["reuse_gpkg_name"]
        renderer = build_renderer(geom, spec["field"], spec["labels"],
                                   spec["palette"], spec["widths"])
        out_lyrx = out_dir / f"{name}.lyrx"
        write_single_lyrx(out_lyrx, f"{existing}.gpkg", existing, title, renderer)
        prev = next((s for s in summary if s["layer"] == existing), None)
        n_features = prev["n_features"] if prev and prev.get("n_features") else None
        summary.append({
            "layer": name, "geom": geom, "field": spec["field"],
            "n_features": n_features, "out_dir": spec["out_dir"],
            "reuse_gpkg_name": existing, "counts": {},
        })
        print(f"  ✓ {spec['out_dir']}/{name}  (lyrx-only, reuses {existing}.gpkg)")
        continue

    gdf = gpd.read_file(str(src))

    # Apply field-aliases (rename source column → desired output column).
    for out_field, src_field in (spec.get("field_aliases") or {}).items():
        if src_field in gdf.columns and out_field not in gdf.columns:
            gdf[out_field] = gdf[src_field]
        elif src_field not in gdf.columns and out_field not in gdf.columns:
            print(f"  ! field alias source missing: {src_field} (for {out_field})")

    if spec["method"] == "multi":
        # Compute all sub class columns
        class_fields = []
        for sub in spec["attributes"]:
            cls = classify_series(
                gdf[sub["field"]], sub["method"],
                sub["labels"], sub.get("bins"),
            )
            gdf[sub["out_field"]] = cls
            class_fields.append(sub["out_field"])

        # Write gpkg (single file)
        out_gpkg = out_dir / f"{name}.gpkg"
        if out_gpkg.exists():
            out_gpkg.unlink()
        gdf.to_file(str(out_gpkg), driver="GPKG", layer=name)

        # Write multi-layer .lyrx
        out_lyrx = out_dir / f"{name}.lyrx"
        write_multi_lyrx(out_lyrx, f"{name}.gpkg", name, title,
                         spec["attributes"], geom)

        summary.append({
            "layer": name, "geom": geom, "field": ", ".join(class_fields),
            "n_features": len(gdf), "out_dir": spec["out_dir"],
            "multi": True,
            "counts": {sub["out_field"]: gdf[sub["out_field"]].value_counts().to_dict()
                       for sub in spec["attributes"]},
        })
        print(f"  ✓ {spec['out_dir']}/{name}  (multi, {len(gdf)} features, "
              f"{len(spec['attributes'])} sub-renderers)")
        continue

    # Single-attribute build path
    if spec["method"] == "precomputed":
        class_field = spec["field"]
    else:
        class_field = f"{name}_class"
        gdf[class_field] = classify_single(gdf, spec)

    out_gpkg = out_dir / f"{name}.gpkg"
    if out_gpkg.exists():
        out_gpkg.unlink()
    gdf.to_file(str(out_gpkg), driver="GPKG", layer=name)

    renderer = build_renderer(geom, class_field, spec["labels"],
                              spec["palette"], spec["widths"])
    out_lyrx = out_dir / f"{name}.lyrx"
    write_single_lyrx(out_lyrx, f"{name}.gpkg", name, title, renderer)

    summary.append({"layer": name, "geom": geom, "field": class_field,
                    "n_features": len(gdf), "out_dir": spec["out_dir"],
                    "counts": gdf[class_field].value_counts().to_dict()})
    print(f"  ✓ {spec['out_dir']}/{name}  ({geom}, {len(gdf)} features → {class_field})")

print(f"\nDone. Wrote {len([s for s in summary if 'n_features' in s])} layer pairs into {BUNDLE}")


## Master atlas — one `.lyrx` for the whole hierarchy


In [ ]:
ROOT_URI         = "CIMPATH=Internal_Map/atlas_root.json"
FINAL_URI        = "CIMPATH=Internal_Map/atlas_final_results.json"
SUBINDICES_URI   = "CIMPATH=Internal_Map/atlas_sub_indices.json"
INTERMEDIATE_URI = "CIMPATH=Internal_Map/atlas_intermediate.json"
HAZARD_URI       = "CIMPATH=Internal_Map/atlas_hazard_exposure.json"
TRAVEL_URI       = "CIMPATH=Internal_Map/atlas_travel_disruption.json"
LOCAL_URI        = "CIMPATH=Internal_Map/atlas_local_accessibility.json"
INPUT_URI        = "CIMPATH=Internal_Map/atlas_input.json"

def _atlas_feat_uri(name: str) -> str:
    return f"CIMPATH=Internal_Map/atlas_feat_{name}.json"

def _atlas_multi_uri(name: str, out_field: str) -> str:
    return f"CIMPATH=Internal_Map/atlas_multi_{name}__{out_field}.json"

def _atlas_multi_group_uri(name: str) -> str:
    return f"CIMPATH=Internal_Map/atlas_multi_{name}_group.json"

# Only the headline composite map is visible at start
HEADLINE_LAYERS = {"total_climate_criticality_short"}

# Collect feature defs + nested-group defs, indexed by manifest name → list[URI]
# (singletons emit one URI; multi specs emit a single sub-group URI plus N children.)
present_specs = [s for s in LAYERS if Path(s["src"]).exists()]
top_uris_per_name: dict[str, str] = {}
atlas_layer_defs = []

for spec in present_specs:
    name = spec["name"]
    sub_dir = spec["out_dir"]
    gpkg_relpath = f"{sub_dir}\\{name}.gpkg"

    if spec["method"] == "multi":
        # Sub-group: one CIMGroupLayer + N CIMFeatureLayer children
        sub_child_uris = []
        for sub in spec["attributes"]:
            sub_uri = _atlas_multi_uri(name, sub["out_field"])
            renderer = build_renderer(spec["geom"], sub["out_field"],
                                       sub["labels"], sub["palette"], sub["widths"])
            atlas_layer_defs.append(_feature_layer_def(
                uri=sub_uri, name=sub["title"],
                layer_name=name, gpkg_relpath=gpkg_relpath,
                renderer=renderer, visible=False,
            ))
            sub_child_uris.append(sub_uri)

        group_uri = _atlas_multi_group_uri(name)
        atlas_layer_defs.append(_group_layer_def(
            uri=group_uri, name=spec["title"],
            child_uris=sub_child_uris, expanded=False,
        ))
        top_uris_per_name[name] = group_uri
        continue

    # Singleton — could either own its gpkg or re-use an existing one
    if spec.get("reuse_gpkg_name"):
        existing = spec["reuse_gpkg_name"]
        gpkg_relpath = f"{sub_dir}\\{existing}.gpkg"
        atlas_layer_name = existing
    else:
        atlas_layer_name = name

    class_field = spec["field"] if spec["method"] == "precomputed" else f"{name}_class"
    renderer = build_renderer(spec["geom"], class_field,
                               spec["labels"], spec["palette"], spec["widths"])
    feat_uri = _atlas_feat_uri(name)
    atlas_layer_defs.append(_feature_layer_def(
        uri=feat_uri, name=spec["title"],
        layer_name=atlas_layer_name, gpkg_relpath=gpkg_relpath,
        renderer=renderer, visible=(name in HEADLINE_LAYERS),
    ))
    top_uris_per_name[name] = feat_uri


def _present_names(sub_dir: str) -> list[str]:
    return [s["name"] for s in present_specs if s["out_dir"] == sub_dir]


# Top-of-Contents order rules
def _haz_order(name: str) -> tuple:
    order = {"network_hazard_exposure": 0,
             "future_floods_network": 1, "future_floods": 2}
    if name in order:
        return (order[name], "")
    if name.startswith("future_precipitation"):
        return (3, name)
    return (4, name)

haz_names = sorted(_present_names("hazard_exposure"), key=_haz_order)
trav_names = _present_names("travel_disruption")

# Local accessibility: TT impacts above baseline accessibility
loc_names = sorted(
    _present_names("local_accessibility"),
    key=lambda n: (0 if "tt_impact" in n else 1, n),
)

# Split final_results entries: composites in 'Final results', H/T/A in 'Sub-indices'
def _atlas_group_of(spec):
    return spec.get("atlas_group") or ("sub_indices" if spec.get("reuse_gpkg_name") else "final_results")

fin_names = sorted(
    [s["name"] for s in present_specs
     if s["out_dir"] == "final_results" and _atlas_group_of(s) == "final_results"],
    key=lambda n: (0 if "short" in n else 1, n),
)

# Sub-indices order: H, T, A (matches the 5d notebook panel order)
sub_idx_order = {"hazard_exposure_index": 0,
                 "travel_disruption_index": 1,
                 "local_accessibility_index": 2}
sub_idx_names = sorted(
    [s["name"] for s in present_specs if _atlas_group_of(s) == "sub_indices"],
    key=lambda n: sub_idx_order.get(n, 99),
)

# Build the group layer defs (containers)
atlas_layer_defs.append(_group_layer_def(
    HAZARD_URI, "Hazard exposure",
    [top_uris_per_name[n] for n in haz_names], expanded=False,
))
atlas_layer_defs.append(_group_layer_def(
    TRAVEL_URI, "National travel disruption",
    [top_uris_per_name[n] for n in trav_names], expanded=False,
))
atlas_layer_defs.append(_group_layer_def(
    LOCAL_URI, "Local accessibility",
    [top_uris_per_name[n] for n in loc_names], expanded=False,
))
input_names = _present_names("input")
atlas_layer_defs.append(_group_layer_def(
    INPUT_URI, "Input",
    [top_uris_per_name[n] for n in input_names], expanded=False,
    description="Raw road-network inputs produced by 1a_Network_Figures.ipynb.",
))
atlas_layer_defs.append(_group_layer_def(
    INTERMEDIATE_URI, "Intermediate results",
    [HAZARD_URI, TRAVEL_URI, LOCAL_URI], expanded=False,
))
atlas_layer_defs.append(_group_layer_def(
    FINAL_URI, "Final results",
    [top_uris_per_name[n] for n in fin_names], expanded=True,
))
atlas_layer_defs.append(_group_layer_def(
    SUBINDICES_URI, "Sub-indices",
    [top_uris_per_name[n] for n in sub_idx_names], expanded=False,
    description="Component sub-indices feeding the composite CC criticality (mirrors the H/T/A panels in 5d).",
))
atlas_layer_defs.append(_group_layer_def(
    ROOT_URI, "Serbia Road Criticality Atlas",
    [FINAL_URI, SUBINDICES_URI, INTERMEDIATE_URI, INPUT_URI],
    description="Composite climate criticality of the Serbian road network — open any sub-layer in the Contents pane.",
    expanded=True,
))

atlas_doc = {
    "type": "CIMLayerDocument",
    "version": "3.0.0",
    "build": 0,
    "layers": [ROOT_URI],
    "layerDefinitions": atlas_layer_defs,
    "binaryReferences": [],
}

atlas_path = BUNDLE / "serbia_criticality_atlas.lyrx"
atlas_path.write_text(json.dumps(atlas_doc, indent=2))

n_groups = sum(1 for d in atlas_layer_defs if d["type"] == "CIMGroupLayer")
n_feats  = sum(1 for d in atlas_layer_defs if d["type"] == "CIMFeatureLayer")
print(f"Wrote master atlas → {atlas_path}")
print(f"  Groups: {n_groups}   Feature layers: {n_feats}   "
      f"Visible at start: {sorted(HEADLINE_LAYERS & set(top_uris_per_name))}")


## Write a README into the bundle


In [ ]:
inventory_rows = []
for r in summary:
    if "n_features" not in r:
        continue
    inventory_rows.append(
        f"| `{r['out_dir']}` | `{r['layer']}` | {r['geom']} | `{r['field']}` | {r['n_features']:,} |"
    )

readme = (
    "# ArcGIS Pro deliverable — Serbia road criticality\n\n"
    f"This folder contains {len(inventory_rows)} layer pairs (`.gpkg` + `.lyrx`) "
    "ready to open in ArcGIS Pro, plus a one-click master atlas.\n\n"
    "## Quickest start — master atlas\n"
    "Drag **`serbia_criticality_atlas.lyrx`** (at the bundle root) into ArcGIS Pro.\n"
    "The Contents pane shows every layer in the hierarchy from `layer_inventory.md`:\n"
    "```\n"
    "Serbia Road Criticality Atlas\n"
    "├── Final results\n"
    "│   ├── Short — total_climate_criticality_short    (visible at start)\n"
    "│   └── Long  — total_climate_criticality_long\n"
    "├── Sub-indices\n"
    "│   ├── H — Hazard exposure\n"
    "│   ├── T — Travel disruption\n"
    "│   └── A — Local accessibility\n"
    "├── Intermediate results\n"
    "│   ├── Hazard exposure (7)\n"
    "│   │   └── Network — combined hazard exposure (group of 4 sub-renderers)\n"
    "│   ├── National travel disruption (1)\n"
    "│   └── Local accessibility (11)\n"
    "└── Input\n"
    "    ├── OSM Road Network — Serbia\n"
    "    └── PERS Road Network — Serbia\n"
    "```\n"
    "All sub-groups start collapsed; only `total_climate_criticality_short` is visible —\n"
    "toggle the rest on as needed. `network_hazard_exposure` itself is a sub-group\n"
    "exposing flood depth, snow drift, pavement temperature and wildfire susceptibility\n"
    "as four separate renderers over the same `.gpkg`.\n\n"
    "## Individual layers\n"
    "Each layer also has its own `.lyrx` next to its `.gpkg`. Drag any of those for a\n"
    "single-layer load. Data connections are by **relative path**, so the whole\n"
    "`ArcGIS_deliverable/` folder is portable.\n\n"
    "## Folder layout\n"
    "```\n"
    "ArcGIS_deliverable/\n"
    "├── serbia_criticality_atlas.lyrx      # master atlas — open this first\n"
    "├── 6_Build_ArcGIS_Deliverable.ipynb   # regenerate everything\n"
    "├── layer_inventory.md                 # full description of every layer\n"
    "├── input/                   # 2 layers  — OSM & PERS road network (from 1a)\n"
    "├── hazard_exposure/         # 7 layers — per-hazard exposure on the road network\n"
    "├── travel_disruption/       # 1 layer  — national-scale SPOF criticality\n"
    "├── local_accessibility/     # 11 layers — baseline accessibility + TT impacts\n"
    "└── final_results/           # 2 layers — final composite climate criticality\n"
    "```\n\n"
    "## Layer inventory\n"
    "| sub-folder | layer | geom | classified field | features |\n"
    "|---|---|---|---|---|\n"
    + "\n".join(inventory_rows)
    + "\n\n"
    "See `layer_inventory.md` for descriptions, attribute columns and source-file\n"
    "renames.\n\n"
    "## Classification methods\n"
    "- **Composite criticality** (`total_climate_criticality_long/short`) — pre-computed\n"
    "  quintile labels `CC_class` / `mean_class` from `5d_Combined_Climate_Criticality.ipynb`.\n"
    "- **`network_hazard_exposure`** — 4 sub-renderers in one `.lyrx`:\n"
    "  flood depth (log-quintile, zero-aware), snow drift (log-quintile, zero-aware),\n"
    "  max pavement temperature (quintile), wildfire susceptibility (binary).\n"
    "- **Travel-time impact** & **accessibility** — 5-class quantile (Very Low … Very\n"
    "  High) computed on-the-fly.\n"
    "- **Future precipitation** — fixed bins on `max_rx1day_pct`:\n"
    "  `[-∞, -2, 0, 5, 10, 20, ∞] %`.\n"
    "- **Future flood RP** — fixed bins on `rp30_mean`: `[0, 25, 50, 100, 150, ∞]`\n"
    "  years. Red = lower RP = higher risk.\n"
    "- **Network criticality** — log-quantile on `phl` (long-tailed distribution).\n"
    "- **Sub-indices** (`hazard_exposure_index`, `travel_disruption_index`,\n"
    "  `local_accessibility_index`) — reuse `total_climate_criticality_long.gpkg`\n"
    "  and render its pre-computed `H_class` / `T_class` / `A_class` quintile fields\n"
    "  with the long-composite purple palette.\n\n"
    "## Regenerate\n"
    "Open `6_Build_ArcGIS_Deliverable.ipynb` and run all cells. The bundle is\n"
    "overwritten in place — the master atlas, README and all layer pairs.\n"
)
(BUNDLE / "README.md").write_text(readme)
print(f"Wrote {BUNDLE / 'README.md'}  ({len(readme):,} bytes)")


## Per-class feature counts (sanity check)


In [ ]:
for r in summary:
    if r.get("multi"):
        print(f"\n— {r['out_dir']}/{r['layer']}  (multi)")
        for field, counts in r["counts"].items():
            print(f"   [{field}]")
            for label, n in counts.items():
                print(f"     {str(label):<22s} {n:>6d}")
        continue
    if "counts" not in r:
        continue
    print(f"\n— {r['out_dir']}/{r['layer']}  ({r['geom']}, field={r['field']})")
    for label, n in r["counts"].items():
        print(f"     {str(label):<22s} {n:>6d}")
